KNN_SVM_프로모션_효율예측분석

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib

In [2]:
mem=pd.read_csv("./data/member.csv")
tran=pd.read_csv("./data/transaction.csv")

* 쇼핑몰 고객 데이터, 프로모션 쿠폰을 발행하고 사용 여부 데이터를 수집
* mem:고객 id, 최근 방문일, 사는 지역, 추천여부, 주요접속채널, 쿠폰사용여부(target)
* tran:고객 id, 구매수량, 총 구매금액
* 전통적 마케팅 분석 방법 rfm 기법을 활용해 고객 데이터에서 파생변수 생성 후 분석
* R: Recency: 현재일-최근 구매일
* F: Frequency: 구매빈도
* M: Monetary: 구매금액
* 종속변수: conversion -> 고객이 프로모션에 반응 했는가? 1=yes, 0=no

In [3]:
mem.head(2)

,id,recency,zip_code,is_referral,channel,conversion
0,906145,10,Surburban,0,Phone,0
1,184478,6,Rural,1,Web,0


In [4]:
tran.head(2)

,id,num_item,total_amount
0,906145,5,34000
1,906145,1,27000


In [5]:
mem.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           64000 non-null  int64 
 1   recency      64000 non-null  int64 
 2   zip_code     64000 non-null  object
 3   is_referral  64000 non-null  int64 
 4   channel      64000 non-null  object
 5   conversion   64000 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 2.9+ MB


In [6]:
mem.describe()

,id,recency,is_referral,conversion
count,64000.000000,64000.000000,64000.000000,64000.000000
mean,550694.137797,5.763734,0.502250,0.146781
std,259105.689773,3.507592,0.499999,0.353890
min,100001.000000,1.000000,0.000000,0.000000
25%,326772.000000,2.000000,0.000000,0.000000
50%,551300.000000,6.000000,1.000000,0.000000
75%,774914.500000,9.000000,1.000000,0.000000
max,999997.000000,12.000000,1.000000,1.000000


In [7]:
data=pd.merge(mem, tran, how='left', on='id')

In [8]:
data

,id,recency,zip_code,is_referral,channel,conversion,num_item,total_amount
0,906145,10,Surburban,0,Phone,0,5,34000
1,906145,10,Surburban,0,Phone,0,1,27000
2,906145,10,Surburban,0,Phone,0,4,33000
3,184478,6,Rural,1,Web,0,4,29000
4,394235,7,Surburban,1,Web,0,4,33000
...,...,...,...,...,...,...,...,...
196831,254229,1,Surburban,0,Web,0,3,33000
196832,254229,1,Surburban,0,Web,0,1,18000
196833,254229,1,Surburban,0,Web,0,3,24000
196834,254229,1,Surburban,0,Web,0,5,14000


In [10]:
data['recency','num_item].groupby(['recency']).count()

SyntaxError: invalid syntax (1421168233.py, line 1)

In [ ]:
mem

In [ ]:
from ydata_profiling import ProfileReport

In [ ]:

profile = ProfileReport(data, title="Selected Features Profiling Report", explorative=True)

profile.to_file("shoppingmall_eda.html")

In [ ]:
data[data.duplicated()]

In [ ]:
data=data.drop_duplicates()

In [ ]:
data

In [ ]:
recency_desc=data.groupby(['recency'])['conversion'].mean()
recency_desc

In [ ]:
num_item_desc=data.groupby(['num_item'])['conversion'].mean()
num_item_desc

In [ ]:
total_amount_desc=data.groupby(['total_amount'])['conversion'].mean()
total_amount_desc

In [ ]:
data['average_amount']=data['total_amount']/data['num_item']

In [ ]:
data

In [ ]:
average_amount_desc=data.groupby(['average_amount'])['conversion'].mean()
average_amount_desc

In [ ]:
data=data.drop(['id'], axis=1)

In [ ]:
data=pd.get_dummies(data, drop_first=True)

In [ ]:
X=data.drop('conversion', axis=1)
y=data['conversion']

In [ ]:
X

In [ ]:
y

In [ ]:
data[data['conversion']==1]

In [ ]:
32471/196836 * 100

In [ ]:
y.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.4, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(X_valid, y_valid, test_size=0.5, stratify=y_valid, random_state=42)

# DecisionTree

In [ ]:
for i in range(1, 11):
    dtc = DecisionTreeClassifier(max_depth=i, class_weight='balanced', random_state=42)
    dtc.fit(X_train, y_train)
    pred = dtc.predict(X_valid)
    print(classification_report(y_valid, pred))
    print("=" * 30, f"test_result {i}", "=" * 30)
    test_pred = dtc.predict(X_test)
    print(classification_report(y_test, test_pred))
    print()

# XGBoost

In [ ]:
xgb=XGBClassifier(scale_pos_weight=5.25, n_jobs=10, random_state=42)
xgb.fit(X_train, y_train)
pred=xgb.predict(X_valid)
print("="*30, "valid_result", "="*30)
print(classification_report(y_valid, pred), end="\n\n")

print("="*30, "test_result", "="*30)
test_pred=xgb.predict(X_test)
print(classification_report(y_test, test_pred), end="\n\n")

# Catboost

In [ ]:
cbc=CatBoostClassifier(class_weights=[1, 5.25], thread_count=10, random_state=42)
cbc.fit(X_train, y_train)
pred=cbc.predict(X_valid)
print("="*30, "valid_result", "="*30)
print(classification_report(y_valid, pred), end="\n\n")

print("="*30, "test_result", "="*30)
test_pred=cbc.predict(X_test)
print(classification_report(y_test, test_pred), end="\n\n")

# lgbm

In [ ]:
lgbm=LGBMClassifier(class_weights="balanced", n_jobs=10, random_state=42)
lgbm.fit(X_train, y_train)
pred=lgbm.predict(X_valid)
print("="*30, "valid_result", "="*30)
print(classification_report(y_valid, pred), end="\n\n")

print("="*30, "test_result", "="*30)
test_pred=lgbm.predict(X_test)
print(classification_report(y_test, test_pred), end="\n\n")

# Randomforest

In [ ]:
rfc= RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=10)
rfc.fit(X_train, y_train)
pred=rfc.predict(X_valid)
print("="*30, "valid_result", "="*30)
print(classification_report(y_valid, pred), end="\n\n")

print("="*30, "test_result", "="*30)
test_pred=rfc.predict(X_test)
print(classification_report(y_test, test_pred), end="\n\n")

# 하이퍼파라미터 튜닝

In [ ]:
dtc = DecisionTreeClassifier(max_depth=1, class_weight='balanced', random_state=42)

In [ ]:
params=dict(criterion=['gini','entropy','log_loss'], 
            max_depth=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            random_state=[42],
            class_weight=[None,'balanced'])

In [ ]:
rand_cv= RandomizedSearchCV(dtc, param_distributions=params, scoring='f1',cv=5, n_jobs=10, random_state=42)

rand_cv.fit(X_train, y_train)
pred=rand_cv.predict(X_valid)
print("="*30, "valid_result", "="*30)
print("best_params: ", rand_cv.best_params_)
print("best_score_: ", rand_cv.best_score_)
print(classification_report(y_valid, pred), end="\n\n")

print("="*30, "test_result", "="*30)
test_pred=rand_cv.predict(X_test)
print(classification_report(y_test, test_pred), end="\n\n")

# feature importance

In [ ]:
best_model = rand_cv.best_estimator_

In [ ]:
importances = best_model.feature_importances_
features = X_train.columns
importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})

In [ ]:
importance_df = importance_df.sort_values(by='Importance', ascending=False)
importance_df

# 데이터 증폭, 축소 해보기

In [ ]:
from imblearn.over_sampling import SMOTENC

In [ ]:
smt= SMOTENC(categorical_features=list(range(4, X_train.shape[1])), random_state=10, k_neighbors=2, n_jobs=-1)
smt_X, smt_y = smt.fit_resample(X_train, y_train)

In [ ]:
len(smt_y)

In [ ]:
len(y)

# 선생님 풀이

In [11]:
mem.isna().sum()

id             0
recency        0
zip_code       0
is_referral    0
channel        0
conversion     0
dtype: int64

In [12]:
tran.isna().sum()

id              0
num_item        0
total_amount    0
dtype: int64

In [13]:
mem.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           64000 non-null  int64 
 1   recency      64000 non-null  int64 
 2   zip_code     64000 non-null  object
 3   is_referral  64000 non-null  int64 
 4   channel      64000 non-null  object
 5   conversion   64000 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 2.9+ MB


In [14]:
tran.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196836 entries, 0 to 196835
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   id            196836 non-null  int64
 1   num_item      196836 non-null  int64
 2   total_amount  196836 non-null  int64
dtypes: int64(3)
memory usage: 4.5 MB


In [15]:
mem.describe()

,id,recency,is_referral,conversion
count,64000.000000,64000.000000,64000.000000,64000.000000
mean,550694.137797,5.763734,0.502250,0.146781
std,259105.689773,3.507592,0.499999,0.353890
min,100001.000000,1.000000,0.000000,0.000000
25%,326772.000000,2.000000,0.000000,0.000000
50%,551300.000000,6.000000,1.000000,0.000000
75%,774914.500000,9.000000,1.000000,0.000000
max,999997.000000,12.000000,1.000000,1.000000


In [16]:
tran.describe()

,id,num_item,total_amount
count,196836.000000,196836.000000,196836.000000
mean,550557.552932,3.078365,21837.102969
std,259254.795613,1.478408,8218.005565
min,100001.000000,1.000000,8000.000000
25%,326719.000000,2.000000,15000.000000
50%,550918.000000,3.000000,22000.000000
75%,774916.000000,4.000000,29000.000000
max,999997.000000,6.000000,38000.000000


In [17]:
mem.columns

Index(['id', 'recency', 'zip_code', 'is_referral', 'channel', 'conversion'], dtype='object')

In [18]:
mem['id'].nunique

<bound method IndexOpsMixin.nunique of 0        906145
1        184478
2        394235
3        130152
4        940352
          ...  
63995    838295
63996    547316
63997    131575
63998    603659
63999    254229
Name: id, Length: 64000, dtype: int64>

In [19]:
mem['recency'].value_counts()

recency
1     8952
10    7565
2     7537
9     6441
3     5904
4     5077
6     4605
5     4510
7     4078
11    3504
8     3495
12    2332
Name: count, dtype: int64

In [20]:
mem['zip_code'].value_counts()

zip_code
Surburban    28776
Urban        25661
Rural         9563
Name: count, dtype: int64

In [21]:
mem['is_referral'].value_counts()

is_referral
1    32144
0    31856
Name: count, dtype: int64

In [22]:
mem['channel'].value_counts()

channel
Web             28217
Phone           28021
Multichannel     7762
Name: count, dtype: int64

In [23]:
mem['conversion'].value_counts()

conversion
0    54606
1     9394
Name: count, dtype: int64

In [24]:
mem.columns

Index(['id', 'recency', 'zip_code', 'is_referral', 'channel', 'conversion'], dtype='object')

In [25]:
tran

,id,num_item,total_amount
0,906145,5,34000
1,906145,1,27000
2,906145,4,33000
3,184478,4,29000
4,394235,4,33000
...,...,...,...
196831,536246,5,24000
196832,927617,5,26000
196833,927617,3,22000
196834,927617,3,18000


In [26]:
tran[['id', 'num_item']].groupby('id').count()

,num_item
id,
100001,2
100008,1
100032,3
100036,5
100070,4
...,...
999932,1
999981,4
999990,3


In [27]:
tran.groupby('id')['num_item'].mean()

id
100001    3.500000
100008    5.000000
100032    2.666667
100036    3.000000
100070    3.250000
            ...   
999932    5.000000
999981    2.000000
999990    3.000000
999995    2.000000
999997    2.000000
Name: num_item, Length: 64000, dtype: float64

In [35]:
mean_item_amount=tran.groupby('id')[['num_item','total_amount']].mean()
mean_item_amount=mean_item_amount.reset_index()
mean_item_amount.columns=['id', 'mean_num_item', 'mean_total_amount']

In [36]:
mean_item_amount

,id,mean_num_item,mean_total_amount
0,100001,3.500000,26000.000000
1,100008,5.000000,26000.000000
2,100032,2.666667,20666.666667
3,100036,3.000000,25800.000000
4,100070,3.250000,21250.000000
...,...,...,...
63995,999932,5.000000,32000.000000
63996,999981,2.000000,22750.000000
63997,999990,3.000000,28000.000000
63998,999995,2.000000,27000.000000


In [37]:
tran.groupby('id')['id'].count()

id
100001    2
100008    1
100032    3
100036    5
100070    4
         ..
999932    1
999981    4
999990    3
999995    1
999997    1
Name: id, Length: 64000, dtype: int64

In [39]:
mean_item_amount=mean_item_amount.set_index('id')

In [40]:
freq=tran.groupby('id')['id'].count()

In [42]:
mean_item_amount=mean_item_amount.join(freq)

In [43]:
mean_item_amount.rename(columns={'id':'frequency'})

,mean_num_item,mean_total_amount,frequency
id,,,
100001,3.500000,26000.000000,2
100008,5.000000,26000.000000,1
100032,2.666667,20666.666667,3
100036,3.000000,25800.000000,5
100070,3.250000,21250.000000,4
...,...,...,...
999932,5.000000,32000.000000,1
999981,2.000000,22750.000000,4
999990,3.000000,28000.000000,3


In [45]:
tran.groupby('id')[['num_item','total_amount']]

In [ ]:
mean_item_amount=mean_item_amount.join(total_num_amount)
mean_item_amount.reset_index()

회원정보 테이블과 구매개수, 금액 등을 그룹연산한 mean_item_amount 합치기

In [ ]:
data=pd.merge(mem, mean_item_amount, how='left', on='id')
data

In [46]:
data['mean_num_item'].plot(kind='hist')

KeyError: 'mean_num_item'

In [ ]:
data['mean_total_amount'].plot(hist='kind')

In [ ]:
data['num_item'].plot(hist='kind')

In [ ]:
data.columns

In [ ]:
col_names=data[['recency','zip_code', 'is_referral','channel','mean_num_item','mean_total_amount'
     , 'frequency','num_item','total_amount']]

In [ ]:
for col in col_names:
    print("="*30, col, "="*30)
    print(data.groupby(col)['conversion'].mean().sort_values(ascending=False))    
    print()

# 거리기반의 알고리즘 사용시 독립변수들 간의 단위를 꼭 맞춰줘야 함
# 스케일링
* Min-Max Scaler: 모든 숫자를 0-1 사이의 숫자로 변환 - 데이터 분포의 모양을 그대로 유지
* Standard Scaler: 평균을 0, 표준편차를 1로 하는 정규분포 형태로 변환- 데이터의 분포 모양이 정규분포로 바뀜 -> 원래 데이터의 분포 특성을 무시하게 됨
* RobustScaler: 사분위수를 이용해서 데이터를 스케일링-데이터에 이상값이 있을 때

# 머신러닝 모델별 스케일러
* knn(최근접이웃): MinMaxScaler, 데이터 이상치가 있는 경우 RobustScaler이용
* SVM(서포트 벡터 머신): StandardScaler, 데이터 이상치가 있는 경우 RobustScaler이용
* Logistic Regression: StandardScaler, 데이터 이상치가 있는 경우 RobustScaler이용
* Linear / Ridge / Lasso: StandardScaler, 데이터 이상치가 있는 경우 RobustScaler이용
* KMeans / DBSCAN: MinMax or StandardScaler
* DecisionTree, RandomForest, XGBoost: 스케일링 불필요, 이상치가 있는 경우에도 안해도 됨
* Naive Bayes: 스케일링 불필요, 이상치가 있는 경우에도 안해도 됨
* 인공신경망: MinMax or StandardScaler, 이상치가 있는 경우 RobustScaler

# 스케일링 시점: train/test로 나눈 후에 실시

In [ ]:
data

카테고리 변수를 one-hot encoding

In [ ]:
data=pd.get_dummies(data, columns=['zip-code','channel'], drop_first=True)

홀드 아웃

In [ ]:
X=data.drop(['id','conversion'], axis=1)
y=data['conversion']

In [ ]:
X

In [ ]:
y.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_valid, y_train, y_valid= train_test_split(X, y, test_size=0.4, stratify=y, randomstate=42)
X_valid, X_test, y_valid, y_test= train_test_split(X_valid, y_valid, test_size=0.4, stratify=y, randomstate=42)

# 데이터 단위 맞추기 위해서 scaling

In [ ]:
from skklearn.preprocessing import StandarddScale, MinMaxScaler, RobustScaler

In [ ]:
mms=MinMaxScaler()
mms.fit(X_train)
mms_X_train = mms.transform(X_train)
mms_X_valid= mms.transform(X_valid)
mms_X_test=mms.transform(X_test)

In [ ]:
mms_X_train=pd.DataFrame(mms_X_train, columns=X_train.columns)

In [ ]:
mms_X_valid=pd.DataFrame(mms_X_valid, columns=X_train.columns)

In [ ]:
mms_X_test=pd.DataFrame(mms_X_test, columns=X_train.columns)

In [ ]:
X_test

In [ ]:
mms_X_test

# KNN(K-Nearest Neighbor)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

In [ ]:
knn= KNeighborsClassifier(n_jobs=8)
knn.fit(mms_X_train, y_train)
valid_pred=knn.predict(mms_X_valid)
print(classification_report(y_valid, valid_pred))

# KNN의 하이퍼파라미터 튜닝
* n_neighbors = ?
* 전체 샘플수가 적을 때 10000개 이하일 때 3, 5, 7 같은 홀수 값
* 전체 샘플수가 10000개 초과시 루트(n), log2(n)


In [ ]:
len(mmms_X_train)

# 최적K는?

In [ ]:
# 루트를 이용한 최적 K 계산
n=len(mms_X_train)
print(n)
k=int(np.sqrt(n))
print(np.sqrt(n),k)

In [ ]:
# log2(n)을 이용한 최적 K 계산
k_log2=int(np.log2(n))
print(k_log2)

In [ ]:
# 루트를 이용해 계산한 k 값 195적용
knn= KNeighborsClassifier(n_neighbors=195, n_jobs=8)
knn.fit(mms_X_train, y_train)
valid_pred=knn.predict(mms_X_valid)
print(classification_report(y_valid, valid_pred))

In [ ]:
# 로그를 이용해 계산한 k 값 15적용
knn= KNeighborsClassifier(n_neighbors=15, n_jobs=8)
knn.fit(mms_X_train, y_train)
valid_pred=knn.predict(mms_X_valid)
print(classification_report(y_valid, valid_pred))

In [ ]:
mms_X_train.columns

In [ ]:
from imblearn.over_sampling import SMOTENC

In [ ]:
smtnc=SMOTENC(categorical_features=[1,7,8,9,10], random_state=42)
smt_X_train, smt_y_train =smtnc.fit_resample(mms_X_train, y_train)

In [ ]:
for i in range(3, 22,2):
    knn= KNeighborsClassifier(n_neighbors=i, n_jobs=8)
    knn.fit(smt_X_train, smt_y_train)
    valid_pred=knn.predict(mms_X_valid)
    print(classification_report(y_valid, valid_pred))
    print(f"=========Test Result{i} ============")
    test_pred=knn.predict(X_test)
    print(classification_report(y_test, test_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rfc= RandomForestClassifier(class_weight="balanced", n_jobs=8, random_state=42)
rfc.fit(smt_X_train, smt_y_train)
valid_pred=rfc.predict(smt_X_valid)
print(classification_report(y_valid, valid_pr

# 서포트 벡터 SVC를 사용해서 분석
* StandardScaler
* 카테고리 변수는 제외하고 StandardScaler 사용하는 것이 좋음

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X_train.columns

In [ ]:
X_train_cat=X_train[['is_referall']]
X_traim_num=X_train

In [ ]:
num_cols=X_train_num.columns

In [ ]:
ss=StandardScaler()
ss.fit(X_train_num)
X_train_num_temp=ss.transform(X_train_num)
X_valid_num_temp=ss.transform(X_valid[num_cols])
X_test_num_temp=ss.transform(X_test[num_cols])

In [ ]:
ss_X_train_num=pd.DataFrame(X_train_num_temp, columns=num_cols, index=X_train_num.index)


In [ ]:
ss_X_train=pd.concat([ss_X_valid_num, X_valid[cat_cols]], axis=1)
ss_X_test=pd.concat([ss_X_test_num, X_test[cat_cols]], axis=1)

In [ ]:
from sklearn.svm import SVC

In [ ]:
svc=SVC()

# 복습